# Helmet Detector Training — Traffic Analytics System

Fine-tune **YOLO11s** on the balanced helmet dataset (~41K train images, **1.1:1** helmet / no_helmet).

**Before running:** add input dataset **`traffic-helmet-balanced`** in the sidebar.

Full steps: see `docs/KAGGLE_HELMET_TRAINING.md` in the repo.

| Setting | Value |
|---------|-------|
| GPU | T4 x1 (Notebook Settings → Accelerator) |
| Internet | On |
| Est. time | 3–5 hours |

In [ ]:
# --- CONFIG (edit if your Kaggle dataset slugs differ) ---
DATASET_SLUG = "traffic-helmet-balanced"   # required input dataset
WEIGHTS_SLUG = "traffic-helmet-weights"      # optional: partial helmet_detector.pt

CONTINUE_FROM_INPUT = False   # True = start from uploaded models/helmet_detector.pt
RESUME = False                # True = resume last.pt from /kaggle/working

BASE_WEIGHTS = "yolo11s.pt"   # COCO pretrained (downloaded if missing)
EPOCHS = 50
BATCH = 16                    # use 8 if CUDA OOM
IMGSZ = 640
PATIENCE = 4
CLOSE_MOSAIC = 10
WORKERS = 2                   # Kaggle CPUs are limited
SEED = 42

RUN_NAME = "helmet_detector"
PROJECT = "traffic_analytics"

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable GPU: Notebook Settings → Accelerator → GPU T4 x1")

!pip install -q ultralytics>=8.3.0

In [ ]:
INPUT_ROOT = Path("/kaggle/input") / DATASET_SLUG / "helmet_detection"
DATA_YAML = INPUT_ROOT / "data.yaml"
WORKING = Path("/kaggle/working")
RUN_DIR = WORKING / "runs" / "detect" / PROJECT / RUN_NAME
LAST_PT = RUN_DIR / "weights" / "last.pt"
BEST_PT = RUN_DIR / "weights" / "best.pt"
OUTPUT_MODEL = WORKING / "helmet_detector.pt"

assert INPUT_ROOT.is_dir(), f"Dataset not found: {INPUT_ROOT}"
assert DATA_YAML.is_file(), f"Missing data.yaml: {DATA_YAML}"

for split in ("train", "valid", "test"):
    img_dir = INPUT_ROOT / split / "images"
    n = sum(1 for _ in img_dir.iterdir()) if img_dir.is_dir() else 0
    print(f"{split}: {n} images")

print("\ndata.yaml:")
print(DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO

# Resolve starting weights
if RESUME and LAST_PT.is_file():
    weights = str(LAST_PT)
    mode = "resume"
elif CONTINUE_FROM_INPUT:
    candidate = Path("/kaggle/input") / WEIGHTS_SLUG / "helmet_detector.pt"
    if not candidate.is_file():
        candidate = Path("/kaggle/input") / WEIGHTS_SLUG / "best.pt"
    if candidate.is_file():
        weights = str(candidate)
        mode = "continue-best"
    else:
        weights = BASE_WEIGHTS
        mode = "fresh (weights dataset missing)"
else:
    weights = BASE_WEIGHTS
    mode = "fresh"

print(f"Mode: {mode}")
print(f"Weights: {weights}")

model = YOLO(weights)

train_kwargs = dict(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    close_mosaic=CLOSE_MOSAIC,
    project=str(WORKING / "runs" / "detect" / PROJECT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    device=0,
    amp=True,
    workers=WORKERS,
    save_period=5,
    seed=SEED,
    cos_lr=True,
    mosaic=1.0,
    plots=True,
    val=True,
    resume=(mode == "resume"),
    # Indian dashcam aug (matches pipeline_config.yaml)
    hsv_h=0.02,
    hsv_s=0.80,
    hsv_v=0.60,
    degrees=10.0,
    translate=0.18,
    scale=0.50,
    fliplr=0.5,
)

results = model.train(**train_kwargs)
print("Training finished.")

In [ ]:
# Copy best weights for easy download
if not BEST_PT.is_file():
    BEST_PT = Path(results.save_dir) / "weights" / "best.pt"

assert BEST_PT.is_file(), f"best.pt not found at {BEST_PT}"
shutil.copy2(BEST_PT, OUTPUT_MODEL)
print(f"Saved -> {OUTPUT_MODEL} ({OUTPUT_MODEL.stat().st_size / 1e6:.1f} MB)")

# Optional: validate on test split
metrics = model.val(data=str(DATA_YAML), split="test")
print("Test mAP50:", metrics.box.map50)

In [ ]:
# Zip full run folder for download (plots, results.csv, weights)
zip_path = WORKING / "helmet_training_output.zip"
run_folder = BEST_PT.parent.parent

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in run_folder.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(WORKING))
    zf.write(OUTPUT_MODEL, OUTPUT_MODEL.name)

print(f"Download from Output tab:")
print(f"  - {OUTPUT_MODEL.name}")
print(f"  - {zip_path.name}")
print("\nOn your PC:")
print("  copy helmet_detector.pt -> models/helmet_detector.pt")
print("  python ai_pipeline.py")